# N3 Slinky Sim Seed Ablation

Runs seed ablations on the 3-node simulated slinky data using the same core settings as `2d_slinky_train_from_sim_LLT.ipynb`. All run outputs, seed summaries, plots, and optional post-training Hessian diagnostics are collected under one `OUTPUT_DIR`.

In [ ]:
import os
from pathlib import Path

import jax
import jax.numpy as jnp

jax.config.update("jax_enable_x64", True)
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")

from properties import SlinkyN3Properties
from run_architectures import SweepConfig

ROOT = Path.cwd()
OUTPUT_DIR = ROOT / "seed_ablation_outputs_n3_slinky_sim"
SUMMARY_DIR = OUTPUT_DIR / "seed_ablation_summary"

train_file = "../simulation_data_2D/3_noded/n3_slinky_sim_train_dataset_7_trajs.npz"
valid_file = "../simulation_data_2D/3_noded/n3_slinky_sim_test_dataset_6_trajs.npz"

properties = SlinkyN3Properties(mass=0.2)
K_init_chol = (0.02, 0.0, 0.05)
K_init_diag = (0.02, 0.05)

base_cfg = SweepConfig(
    der_K_diag=K_init_diag,
    der_K_chol=K_init_chol,
    hidden=(10, 10),
    corr_factor=0.05,
    input_mode="invariant",
    only_stretching_NN=True,
    only_bending_NN=False,
    zero_reference=True,
    activation="tanh",
    n_epochs=500,
    lr=1e-3,
    seed=42,
    seed_list=tuple(range(26)) # for full seed ablation use 0-25
    valid_every=1,
    max_dlambda=1e-2,
    iters=20,
    ls_steps=10,
    abs_tol=1e-4,
    rel_tol=1e-4,
    early_stop=True,
    train_fail_on_nonconvergence=True,
    prediction_fail_on_nonconvergence=False,
    hessian_reg_strength=1e-4,
    hessian_reg_probes=1,
    hessian_reg_seed=0,
    force_key="F",
    force_loss_strength=0.1,
    force_components=(0,),
    force_sign=1.0,
    return_loss_components=True,
    early_stopping=True,
    early_stopping_patience=200,
    early_stopping_min_delta=1e-5,
    restore_best_model=True,
    output_dir=str(OUTPUT_DIR),
    save_npz=True,
    save_model=True,
    save_plots=True,
    save_force_predictions=True,
    plot_force_predictions=True,
    save_hessian_diagnostics=False,
    save_energy_landscapes=True,
    energy_snapshot_initial=True,
    energy_snapshot_final=True,
    energy_snapshot_epochs=(),
    energy_snapshot_every=None,
    energy_snapshot_use_valid=True,
    energy_snapshot_dpi=180,
    energy_snapshot_n_grid=None,
    verbose=True,
    continue_on_failure=True,
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Saving seed ablation results under: {OUTPUT_DIR.resolve()}")

Saving seed ablation results under: /Users/radha/GitRepos/dismech-jax/examples/slinky/slinky_2D/seed_ablation_outputs_n3_slinky_sim


In [ ]:
from run_architectures import subset_all, subset_main_paper_candidates

# Default: a focused seed ablation on the main comparison candidates.
selected_architectures = subset_main_paper_candidates()

# To run every registered architecture, use:
# selected_architectures = subset_all()

print(f"Running {len(selected_architectures)} architectures across seeds {base_cfg.seed_list}:")
for name in selected_architectures:
    print(f"  - {name}")

Running 1 architectures across seeds (42,):
  - diag_energy_mlp


In [3]:
from seed_ablation_utils import run_seed_ablation

all_seed_results = run_seed_ablation(
    properties=properties,
    train_file=train_file,
    valid_file=valid_file,
    base_cfg=base_cfg,
    selected_architectures=selected_architectures,
)


SEED ABLATION for architecture: diag_energy_mlp

--- Running seed 42 for diag_energy_mlp ---

Running architecture: diag_energy_mlp
  model_cls               : DiagonalPlusEnergyNN
  which_case              : MLP
  hidden                  : (10, 10)
  input_mode              : invariant
  only_stretching_NN      : True
  only_bending_NN         : False
  activation              : tanh
  corr_factor             : 0.05
  zero_reference          : True
  seed                    : 42
  max_dlambda             : 0.01
  iters                   : 20
  ls_steps                : 10
  abs_tol                 : 0.0001
  rel_tol                 : 0.0001
  early_stop              : True
  training fail_on_nonconvergence       : True
  validation loss fail_on_nonconvergence: False
  prediction fail_on_nonconvergence     : False
  early_stopping          : True
  early_stopping_patience : 200
  restore_best_model      : True
  hessian_reg_strength    : 0.0001
  hessian_reg_probes      : 1
  hessian_

/Users/radha/GitRepos/dismech-jax/examples/slinky/slinky_2D/architecture_plots.py:234: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()


In [4]:
from seed_ablation_utils import (
    make_seed_ablation_plots,
    print_seed_summary,
    save_seed_summary_json,
)

print_seed_summary(all_seed_results)
save_seed_summary_json(all_seed_results, output_dir=str(SUMMARY_DIR))

make_seed_ablation_plots(
    all_seed_results=all_seed_results,
    output_dir=str(SUMMARY_DIR),
    traj_idx=0,
    x_idx=4,
    z_idx=6,
)

print("Seed ablation summary plots written to:", SUMMARY_DIR.resolve())


##############################################################################################################
SEED ABLATION SUMMARY
##############################################################################################################

Architecture: diag_energy_mlp
  n_seeds           : 1
  n_success         : 1
  n_failed          : 0
  best_seed         : 42
  best final valid  : 6.220572e-02
  worst final valid : 6.220572e-02
  mean final valid  : 6.220572e-02
  std  final valid  : 0.000000e+00
  median final valid: 6.220572e-02
  mean final train  : 8.487649e-02
  std  final train  : 0.000000e+00
  mean valid disp   : 1.397401e-02
  mean valid force  : 4.823170e-01
Seed ablation summary plots written to: /Users/radha/GitRepos/dismech-jax/examples/slinky/slinky_2D/seed_ablation_outputs_n3_slinky_sim/seed_ablation_summary


In [5]:
from seed_ablation_utils import run_seed_hessian_diagnostics

# Post-training Hessian diagnostics. This is separate from training so the
# seed sweep stays focused on optimization/runtime behavior.
HESSIAN_USE_PREDICTED = True
HESSIAN_STRIDE = 10
HESSIAN_MAX_TRAJECTORIES = 1  # set all_trajectories=True below to process every trajectory

run_seed_hessian_diagnostics(
    str(OUTPUT_DIR),
    use_predicted=HESSIAN_USE_PREDICTED,
    splits=("train", "valid"),
    stride=HESSIAN_STRIDE,
    max_trajectories=HESSIAN_MAX_TRAJECTORIES,
    all_trajectories=False,
    fail_on_nonconvergence=False,
)

[ok] /Users/radha/GitRepos/dismech-jax/examples/slinky/slinky_2D/seed_ablation_outputs_n3_slinky_sim/diag_energy_mlp__hid_10x10__inp_invariant__stretchNN_1__bendNN_0__act_tanh__corr_0.05__zr1__seed_42__mdl_0.01__it_20__hreg_0.0001__hprobe_1__hseed_0__floss_0.1__fcomp_0__fsign_1 | train: M=2.578e-01, kappa=5.890e+00, states=15 | valid: M=2.608e-01, kappa=6.127e+00, states=15
Wrote Hessian diagnostics table with 2 rows to /Users/radha/GitRepos/dismech-jax/examples/slinky/slinky_2D/seed_ablation_outputs_n3_slinky_sim/hessian_diagnostics_table.csv
Finished Hessian diagnostics for 1/1 seed runs.


1

In [6]:
print("All seed ablation artifacts are organized under:")
print(" ", OUTPUT_DIR.resolve())
print("\nMain subfolders/files to inspect:")
print("  - <architecture>__...__seed_<n>/results.npz")
print("  - <architecture>__...__seed_<n>/model.eqx")
print("  - <architecture>__...__seed_<n>/loss_curves.png")
print("  - <architecture>__...__seed_<n>/force_pred_vs_truth_*.png")
print("  - <architecture>__...__seed_<n>/energy_landscapes/")
print("  - <architecture>__...__seed_<n>/hessian_diagnostics_*.npz")
print("  - seed_ablation_summary/")

All seed ablation artifacts are organized under:
  /Users/radha/GitRepos/dismech-jax/examples/slinky/slinky_2D/seed_ablation_outputs_n3_slinky_sim

Main subfolders/files to inspect:
  - <architecture>__...__seed_<n>/results.npz
  - <architecture>__...__seed_<n>/model.eqx
  - <architecture>__...__seed_<n>/loss_curves.png
  - <architecture>__...__seed_<n>/force_pred_vs_truth_*.png
  - <architecture>__...__seed_<n>/energy_landscapes/
  - <architecture>__...__seed_<n>/hessian_diagnostics_*.npz
  - seed_ablation_summary/
